# CartFlow (P06) — Week 8: Gold Hand-off to Power BI

**Official notebook:** `notebooks/06_powerbi_export.ipynb`

**Purpose:** Validate the approved CartFlow Gold batch summaries, define the controlled Power BI hand-off, prove the hand-off preserves the Gold grain and selected business measures, and document the Power BI connection boundary.

**Gold-only rule:** Power BI must use approved Gold outputs only. Do not connect to raw, Bronze, Silver Candidate, Trusted Silver detail, or Quarantine data.

**Selected Gold batch summaries**
- `agg_sales_daily`
- `agg_seller_performance`
- `agg_category_sales`
- `agg_delivery_delay`
- `agg_payment_review`

Each Gold summary remains an independent source. Do not flatten the five Gold grains into one master table.


## Week 8 workflow

1. Gold source register
2. Gold hand-off validation
3. Controlled Power BI hand-off definitions
4. Gold-to-hand-off reconciliation
5. Repeat-run business-content fingerprint
6. Power BI connection manifest
7. Final hand-off checkpoint

**Important:** PASS/FAIL conclusions must come from actual Databricks execution. Do not manually enter results.


In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT current_catalog() AS active_catalog,
       current_schema() AS active_schema;


active_catalog,active_schema
workspace,default


## 1. Gold source register

| Gold table | Declared grain | Key | Main measures / purpose | Power BI use |
|---|---|---|---|---|
| `agg_sales_daily` | One row per sales date | `sales_date` | `total_orders`, `non_cancelled_orders`, `gmv`, `freight`, `aov` | Sales overview and daily trend |
| `agg_seller_performance` | One row per seller | `seller_id` | `order_count`, `cancelled_orders`, `returned_orders`, `gmv`, `average_review_score` | Seller performance |
| `agg_category_sales` | One row per month + category | `year_month + category_code` | `item_lines`, `order_count`, `gmv`, `item_total`, `freight`, `average_order_value` | Category sales |
| `agg_delivery_delay` | One row per purchase date + customer state | `purchase_date + customer_state` | `delivery_eligible_orders`, `on_time_orders`, `average_delay_days`, `on_time_delivery_rate` | Delivery performance |
| `agg_payment_review` | One row per currency | `currency_code` | `reconciled_orders`, `complete_coverage_orders`, `payment_reconciliation_rate`, `average_review_score`, `reviewed_order_coverage` | Payment/review quality |


In [0]:
%sql
SHOW TABLES LIKE 'agg_*';


database,tableName,isTemporary
default,agg_category_sales,false
default,agg_delivery_delay,false
default,agg_payment_review,false
default,agg_sales_daily,false
default,agg_seller_performance,false


In [0]:
%sql
SELECT 'agg_sales_daily' AS gold_table, COUNT(*) AS rows FROM agg_sales_daily
UNION ALL SELECT 'agg_seller_performance', COUNT(*) FROM agg_seller_performance
UNION ALL SELECT 'agg_category_sales', COUNT(*) FROM agg_category_sales
UNION ALL SELECT 'agg_delivery_delay', COUNT(*) FROM agg_delivery_delay
UNION ALL SELECT 'agg_payment_review', COUNT(*) FROM agg_payment_review
ORDER BY gold_table;


gold_table,rows
agg_category_sales,251
agg_delivery_delay,6700
agg_payment_review,1
agg_sales_daily,335
agg_seller_performance,2939


### Source-register checkpoint

Confirm from the results:
- `workspace.default` is the intended Gold location.
- All five approved batch-summary tables exist.
- Table names match the Week-7 Gold model.
- Row counts are actual Databricks results.

Do not manually type or invent row counts.


## 2. Gold hand-off validation

Verify that:
- report-facing columns exist;
- declared grain keys are unique;
- declared keys are not null/blank;
- selected business measures are populated where expected.


In [0]:
%sql
WITH expected AS (
  SELECT 'agg_sales_daily' AS gold_table, column_name FROM VALUES
    ('sales_date'),('total_orders'),('non_cancelled_orders'),('gmv'),('freight'),('aov') AS v(column_name)
  UNION ALL SELECT 'agg_seller_performance', column_name FROM VALUES
    ('seller_id'),('order_count'),('cancelled_orders'),('returned_orders'),('gmv'),('average_review_score') AS v(column_name)
  UNION ALL SELECT 'agg_category_sales', column_name FROM VALUES
    ('year_month'),('category_code'),('item_lines'),('order_count'),('gmv'),('item_total'),('freight'),('average_order_value') AS v(column_name)
  UNION ALL SELECT 'agg_delivery_delay', column_name FROM VALUES
    ('purchase_date'),('customer_state'),('delivery_eligible_orders'),('on_time_orders'),('average_delay_days'),('on_time_delivery_rate') AS v(column_name)
  UNION ALL SELECT 'agg_payment_review', column_name FROM VALUES
    ('currency_code'),('reconciled_orders'),('complete_coverage_orders'),('payment_reconciliation_rate'),('average_review_score'),('reviewed_order_coverage') AS v(column_name)
),
actual AS (
  SELECT table_name AS gold_table, column_name
  FROM information_schema.columns
  WHERE table_schema=current_schema()
    AND table_name IN ('agg_sales_daily','agg_seller_performance','agg_category_sales','agg_delivery_delay','agg_payment_review')
)
SELECT e.gold_table,e.column_name AS expected_column,
       CASE WHEN a.column_name IS NOT NULL THEN 'PRESENT' ELSE 'MISSING' END AS column_status
FROM expected e LEFT JOIN actual a
ON e.gold_table=a.gold_table AND e.column_name=a.column_name
ORDER BY e.gold_table,e.column_name;


gold_table,expected_column,column_status
agg_category_sales,average_order_value,PRESENT
agg_category_sales,category_code,PRESENT
agg_category_sales,freight,PRESENT
agg_category_sales,gmv,PRESENT
agg_category_sales,item_lines,PRESENT
agg_category_sales,item_total,PRESENT
agg_category_sales,order_count,PRESENT
agg_category_sales,year_month,PRESENT
agg_delivery_delay,average_delay_days,PRESENT
agg_delivery_delay,customer_state,PRESENT


In [0]:
%sql
SELECT 'agg_sales_daily' AS gold_table,COUNT(*) AS rows,COUNT(DISTINCT sales_date) AS distinct_keys,
COUNT(*)-COUNT(DISTINCT sales_date) AS duplicate_rows FROM agg_sales_daily
UNION ALL SELECT 'agg_seller_performance',COUNT(*),COUNT(DISTINCT seller_id),COUNT(*)-COUNT(DISTINCT seller_id) FROM agg_seller_performance
UNION ALL SELECT 'agg_category_sales',COUNT(*),COUNT(DISTINCT CONCAT(COALESCE(CAST(year_month AS STRING),''),'||',COALESCE(CAST(category_code AS STRING),''))),
COUNT(*)-COUNT(DISTINCT CONCAT(COALESCE(CAST(year_month AS STRING),''),'||',COALESCE(CAST(category_code AS STRING),''))) FROM agg_category_sales
UNION ALL SELECT 'agg_delivery_delay',COUNT(*),COUNT(DISTINCT CONCAT(COALESCE(CAST(purchase_date AS STRING),''),'||',COALESCE(CAST(customer_state AS STRING),''))),
COUNT(*)-COUNT(DISTINCT CONCAT(COALESCE(CAST(purchase_date AS STRING),''),'||',COALESCE(CAST(customer_state AS STRING),''))) FROM agg_delivery_delay
UNION ALL SELECT 'agg_payment_review',COUNT(*),COUNT(DISTINCT currency_code),COUNT(*)-COUNT(DISTINCT currency_code) FROM agg_payment_review
ORDER BY gold_table;


gold_table,rows,distinct_keys,duplicate_rows
agg_category_sales,251,251,0
agg_delivery_delay,6700,6700,0
agg_payment_review,1,1,0
agg_sales_daily,335,335,0
agg_seller_performance,2939,2939,0


In [0]:
%sql
SELECT 'agg_sales_daily' AS gold_table,SUM(CASE WHEN sales_date IS NULL THEN 1 ELSE 0 END) AS null_key_rows FROM agg_sales_daily
UNION ALL SELECT 'agg_seller_performance',SUM(CASE WHEN seller_id IS NULL OR TRIM(CAST(seller_id AS STRING))='' THEN 1 ELSE 0 END) FROM agg_seller_performance
UNION ALL SELECT 'agg_category_sales',SUM(CASE WHEN year_month IS NULL OR category_code IS NULL OR TRIM(CAST(category_code AS STRING))='' THEN 1 ELSE 0 END) FROM agg_category_sales
UNION ALL SELECT 'agg_delivery_delay',SUM(CASE WHEN purchase_date IS NULL OR customer_state IS NULL OR TRIM(CAST(customer_state AS STRING))='' THEN 1 ELSE 0 END) FROM agg_delivery_delay
UNION ALL SELECT 'agg_payment_review',SUM(CASE WHEN currency_code IS NULL OR TRIM(CAST(currency_code AS STRING))='' THEN 1 ELSE 0 END) FROM agg_payment_review
ORDER BY gold_table;


gold_table,null_key_rows
agg_category_sales,0
agg_delivery_delay,0
agg_payment_review,0
agg_sales_daily,0
agg_seller_performance,0


### Validation checkpoint

Expected structural results:
- required columns = `PRESENT`;
- duplicate grain rows = `0`;
- null/blank key rows = `0`.

If a check fails, investigate the upstream Gold output. Do not manually edit the hand-off.


## 3. Controlled Power BI hand-off definitions

Keep each source separate and preserve its declared Gold grain.


In [0]:
%sql
SELECT sales_date,total_orders,non_cancelled_orders,gmv,freight,aov
FROM agg_sales_daily ORDER BY sales_date;


sales_date,total_orders,non_cancelled_orders,gmv,freight,aov
2025-01-01,299,286,512201.77,47982.70,1790.91527972
2025-01-02,275,254,217509.26,42134.52,856.33566929
2025-01-03,320,310,260251.31,50850.56,839.52035484
2025-01-04,306,288,490434.84,46274.27,1702.89875000
2025-01-05,279,263,220232.43,41885.31,837.38566540
2025-01-06,294,280,496978.92,47659.25,1774.92471429
2025-01-07,294,278,478427.54,44645.51,1720.96237410
2025-01-08,296,279,738745.74,44692.97,2647.83419355
2025-01-09,290,274,221805.00,45254.33,809.50729927
2025-01-10,277,265,466613.96,45050.73,1760.80739623


In [0]:
%sql
SELECT seller_id,order_count,cancelled_orders,returned_orders,gmv,average_review_score
FROM agg_seller_performance ORDER BY seller_id;


seller_id,order_count,cancelled_orders,returned_orders,gmv,average_review_score
SLR000001,39,1,3,33010.75,3.75
SLR000002,46,1,1,25062.18,3.4634146341463414
SLR000003,33,1,4,22102.66,3.68
SLR000004,42,0,3,278342.66,4.055555555555555
SLR000005,41,6,1,29049.46,3.8666666666666667
SLR000006,38,3,2,272284.95,4.09375
SLR000007,36,2,1,20331.48,3.814814814814815
SLR000008,39,4,0,274100.10,3.806451612903226
SLR000009,48,3,3,32262.31,3.8529411764705883
SLR000010,30,4,0,13504.23,4.260869565217392


In [0]:
%sql
SELECT year_month,category_code,item_lines,order_count,gmv,item_total,freight,average_order_value
FROM agg_category_sales ORDER BY year_month,category_code;


year_month,category_code,item_lines,order_count,gmv,item_total,freight,average_order_value
2025-01,appliances,547,543,614939.26,614939.26,72823.65,1175.792084130019120459
2025-01,automotive,537,537,598498.76,598498.76,72640.10,1162.133514563106796117
2025-01,baby,526,521,619424.57,619424.57,71701.45,1238.849140000000000000
2025-01,beauty,582,578,887603.95,887603.95,77190.27,1610.896460980036297641
2025-01,books,527,526,333599.92,333599.92,65675.07,665.868103792415169661
2025-01,electronics,531,527,587991.86,587991.86,69230.76,1183.082213279678068410
2025-01,fashion,504,498,362697.08,362697.08,67305.69,760.371236897274633124
2025-01,footwear,538,529,352516.76,352516.76,68918.75,707.864979919678714859
2025-01,furniture,478,471,1067757.27,1067757.27,62582.50,2367.532749445676274945
2025-01,garden,568,561,401418.46,401418.46,77353.35,755.966967984934086629


In [0]:
%sql
SELECT purchase_date,customer_state,delivery_eligible_orders,on_time_orders,average_delay_days,on_time_delivery_rate
FROM agg_delivery_delay ORDER BY purchase_date,customer_state;


purchase_date,customer_state,delivery_eligible_orders,on_time_orders,average_delay_days,on_time_delivery_rate
2025-01-01,AP,15,3,1.2,0.2
2025-01-01,BR,15,4,1.2666666666666666,0.26666666666666666
2025-01-01,CG,7,4,0.8571428571428571,0.5714285714285714
2025-01-01,DL,10,4,0.9,0.4
2025-01-01,GA,16,6,1.0,0.375
2025-01-01,GJ,24,11,0.9583333333333334,0.4583333333333333
2025-01-01,HR,10,3,0.6,0.3
2025-01-01,JH,5,2,0.6,0.4
2025-01-01,KA,13,6,1.3846153846153846,0.46153846153846156
2025-01-01,KL,13,6,0.0,0.46153846153846156


In [0]:
%sql
SELECT currency_code,reconciled_orders,complete_coverage_orders,payment_reconciliation_rate,average_review_score,reviewed_order_coverage
FROM agg_payment_review ORDER BY currency_code;


currency_code,reconciled_orders,complete_coverage_orders,payment_reconciliation_rate,average_review_score,reviewed_order_coverage
INR,93808,93841,0.9996483413433361,3.7965113375287185,0.9035810185707516


## 4. Gold-to-hand-off reconciliation

For the direct Databricks Power BI connection, the governed hand-off is the approved Gold table itself. These checks establish the baseline values Power BI must reproduce.


In [0]:
%sql
SELECT 'agg_sales_daily' AS gold_source,COUNT(*) AS gold_rows,COUNT(*) AS handoff_rows,0 AS row_count_difference FROM agg_sales_daily
UNION ALL SELECT 'agg_seller_performance',COUNT(*),COUNT(*),0 FROM agg_seller_performance
UNION ALL SELECT 'agg_category_sales',COUNT(*),COUNT(*),0 FROM agg_category_sales
UNION ALL SELECT 'agg_delivery_delay',COUNT(*),COUNT(*),0 FROM agg_delivery_delay
UNION ALL SELECT 'agg_payment_review',COUNT(*),COUNT(*),0 FROM agg_payment_review
ORDER BY gold_source;


gold_source,gold_rows,handoff_rows,row_count_difference
agg_category_sales,251,251,0
agg_delivery_delay,6700,6700,0
agg_payment_review,1,1,0
agg_sales_daily,335,335,0
agg_seller_performance,2939,2939,0


In [0]:
%sql
SELECT 'agg_sales_daily' AS gold_source,COUNT(*) AS rows,SUM(total_orders) AS total_orders,SUM(non_cancelled_orders) AS non_cancelled_orders,SUM(gmv) AS total_gmv,SUM(freight) AS total_freight FROM agg_sales_daily
UNION ALL SELECT 'agg_seller_performance',COUNT(*),SUM(order_count),SUM(cancelled_orders),SUM(returned_orders),SUM(gmv) FROM agg_seller_performance
UNION ALL SELECT 'agg_category_sales',COUNT(*),SUM(item_lines),SUM(order_count),SUM(gmv),SUM(item_total) FROM agg_category_sales
UNION ALL SELECT 'agg_delivery_delay',COUNT(*),SUM(delivery_eligible_orders),SUM(on_time_orders),SUM(average_delay_days),SUM(on_time_delivery_rate) FROM agg_delivery_delay
UNION ALL SELECT 'agg_payment_review',COUNT(*),SUM(reconciled_orders),SUM(complete_coverage_orders),SUM(payment_reconciliation_rate),SUM(reviewed_order_coverage) FROM agg_payment_review
ORDER BY gold_source;


gold_source,rows,total_orders,non_cancelled_orders,total_gmv,total_freight
agg_category_sales,251,115827,114818,1.4832397195E8,1.4832397195E8
agg_delivery_delay,6700,88634,32225,815.6948847788822,2437.5135386280344
agg_payment_review,1,93808,93841,0.9996483413433361,0.9035810185707516
agg_sales_daily,335,98150,93421,1.4832397195E8,1.546466926E7
agg_seller_performance,2939,115822,5550,5749.0,1.4832397195E8


## 5. Repeat-run business-content fingerprint

Run this cell once. Then rerun the unchanged Gold snapshot. The row counts and business fingerprints should remain unchanged.

Do not manually enter a matching value.


In [0]:
%sql
SELECT 'agg_sales_daily' AS gold_source,COUNT(*) AS rows,
SUM(CAST(xxhash64(CAST(sales_date AS STRING),CAST(total_orders AS STRING),CAST(non_cancelled_orders AS STRING),CAST(gmv AS STRING),CAST(freight AS STRING),CAST(aov AS STRING)) AS DECIMAL(38,0))) AS business_fingerprint
FROM agg_sales_daily
UNION ALL SELECT 'agg_seller_performance',COUNT(*),
SUM(CAST(xxhash64(CAST(seller_id AS STRING),CAST(order_count AS STRING),CAST(cancelled_orders AS STRING),CAST(returned_orders AS STRING),CAST(gmv AS STRING),CAST(average_review_score AS STRING)) AS DECIMAL(38,0))) FROM agg_seller_performance
UNION ALL SELECT 'agg_category_sales',COUNT(*),
SUM(CAST(xxhash64(CAST(year_month AS STRING),CAST(category_code AS STRING),CAST(item_lines AS STRING),CAST(order_count AS STRING),CAST(gmv AS STRING),CAST(item_total AS STRING),CAST(freight AS STRING),CAST(average_order_value AS STRING)) AS DECIMAL(38,0))) FROM agg_category_sales
UNION ALL SELECT 'agg_delivery_delay',COUNT(*),
SUM(CAST(xxhash64(CAST(purchase_date AS STRING),CAST(customer_state AS STRING),CAST(delivery_eligible_orders AS STRING),CAST(on_time_orders AS STRING),CAST(average_delay_days AS STRING),CAST(on_time_delivery_rate AS STRING)) AS DECIMAL(38,0))) FROM agg_delivery_delay
UNION ALL SELECT 'agg_payment_review',COUNT(*),
SUM(CAST(xxhash64(CAST(currency_code AS STRING),CAST(reconciled_orders AS STRING),CAST(complete_coverage_orders AS STRING),CAST(payment_reconciliation_rate AS STRING),CAST(average_review_score AS STRING),CAST(reviewed_order_coverage AS STRING)) AS DECIMAL(38,0))) FROM agg_payment_review
ORDER BY gold_source;


gold_source,rows,business_fingerprint
agg_category_sales,251,40196483940862991765
agg_delivery_delay,6700,537683826677215157871
agg_payment_review,1,240341744766192727
agg_sales_daily,335,-85439454979492955762
agg_seller_performance,2939,-6970189872206464929


## 6. Power BI connection manifest

Connect Power BI only to these five approved Gold tables in `workspace.default`:

- `agg_sales_daily`
- `agg_seller_performance`
- `agg_category_sales`
- `agg_delivery_delay`
- `agg_payment_review`

Keep them separate.

Do not create a combined master Gold table or direct summary-to-summary relationships merely because field names are similar.


## 7. Final hand-off checkpoint

Before treating the Gold hand-off as complete, confirm actual Databricks results show:
- all five Gold tables exist;
- required columns are present;
- declared grain keys are unique;
- keys have no null/blank values;
- selected measures are recorded;
- row counts reconcile;
- repeat-run fingerprints remain unchanged.

### Evidence to capture
Capture readable screenshots of:
1. Gold source inventory and row counts;
2. hand-off validation;
3. Gold-to-hand-off reconciliation;
4. repeat-run fingerprint.

Then preserve the Power BI model and dashboard evidence separately.

**Do not mark a check PASS unless the Databricks result actually supports it.**


In [0]:
%sql
SELECT
    SUM(total_orders) AS gold_total_orders
FROM agg_sales_daily
WHERE sales_date BETWEEN '2025-06-01' AND '2025-12-01';

gold_total_orders
54055


In [0]:
%sql
SELECT
    SUM(reconciled_orders) AS reconciled_orders,
    SUM(complete_coverage_orders) AS complete_coverage_orders,
    SUM(reconciled_orders) * 1.0
        / NULLIF(SUM(complete_coverage_orders), 0) AS payment_reconciliation_rate
FROM agg_payment_review;

reconciled_orders,complete_coverage_orders,payment_reconciliation_rate
93808,93841,0.9996483413433361


In [0]:
%sql
SELECT
    seller_id,
    order_count
FROM agg_seller_performance
ORDER BY order_count DESC
LIMIT 10;

seller_id,order_count
SLR002740,64
SLR002880,64
SLR001569,63
SLR000895,61
SLR001051,60
SLR000147,60
SLR002066,59
SLR000763,59
SLR001282,59
SLR000903,58


In [0]:
%sql
SELECT
    SUM(total_orders) AS gold_total_orders
FROM agg_sales_daily;

gold_total_orders
98150
